In [ ]:
import os
from pathlib import Path
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import seaborn as sns
from IPython.display import display, Markdown

import sim_ranking as sr
import ml_tools as mlt

warnings.simplefilter(action='ignore', category=FutureWarning)

In [ ]:
results_dir = Path("/Users/claudy/dev/work/data/sim_ranking/results/gnn/simple/1128_1127_cv_test_4_8_300Epochs_EdgeUpdating_Dropout0.25_NZGMDB4p1")
wdata = Path("/Users/claudy/dev/work/data")

In [ ]:
run_config = sr.ml.gnn_gm.RunConfig.from_yaml(results_dir / "run_config.yaml")

In [ ]:
# Load observed data to get metadata
nzgmdb_ffp = wdata / run_config.rel_obs_data_ffp
obs_data = sr.data.load_obs_nzgmdb(nzgmdb_ffp)

In [ ]:
# Distance matrix
dist_matrix = sr.utils.calculate_distance_matrix(obs_data.sites, obs_data.site_df)

In [ ]:
# Load attention coefficients
val_attn_coefs = pd.read_parquet(results_dir / "val_attn_coeffs.parquet")

In [ ]:
# Add site-to-site distance
row_ind = dist_matrix.index.get_indexer_for(val_attn_coefs.site_int)
col_ind = dist_matrix.columns.get_indexer_for(val_attn_coefs.obs_site)
val_attn_coefs["dist"] = dist_matrix.values[row_ind, col_ind]

In [ ]:
val_attn_coefs.head()

In [ ]:
n_cv_iters = len(val_attn_coefs.cv_iter.unique())
conv_cols = [cur_col for cur_col in val_attn_coefs.columns if cur_col.startswith("conv_")]


fig, axs = mlt.plotting.get_fig_axes(n_cv_iters, 2, -1, (8, 6))

for ix, (cur_cv_iter, cur_ax) in enumerate(zip(val_attn_coefs.cv_iter.unique(), axs)):
	cur_data = val_attn_coefs.loc[val_attn_coefs.cv_iter == cur_cv_iter]

	for cur_conv in conv_cols:
	    cur_ax.scatter(cur_data.dist, cur_data[cur_conv], label=cur_conv, s=2.5)

	# if ix == 0:
	cur_ax.legend()

	cur_ax.set_xlabel("Site-to-site distance (km)")
	cur_ax.set_ylabel("Attention coefficient")
	cur_ax.grid(linewidth=0.5, alpha=0.5, linestyle="--")
	cur_ax.set_title(f"{cur_cv_iter}")
	cur_ax.set_ylim(0, 1)

fig.tight_layout()

In [ ]:
fig, (ax1, ax2) = mlt.plotting.get_fig_axes(2, 2, 1, (8, 6))

sns.scatterplot(data=val_attn_coefs, x="dist", y="conv_0", ax=ax1, hue="cv_iter", legend=False, size=1.0)
ax1.set_xlabel("Site-to-site distance (km)")
ax1.set_ylabel("Attention coefficient")
ax1.grid(linewidth=0.5, alpha=0.5, linestyle="--")
ax1.set_title("Graph Convolution 0")

sns.scatterplot(data=val_attn_coefs, x="dist", y="conv_1", ax=ax2, hue="cv_iter", legend=False, size=1.0)
ax2.set_xlabel("Site-to-site distance (km)")
ax2.set_ylabel("Attention coefficient")
ax2.grid(linewidth=0.5, alpha=0.5, linestyle="--")
ax2.set_title("Graph Convolution 1")

fig.tight_layout()

In [ ]:
val_attn_coefs.cv_iter.unique()

In [ ]:
val_attn_coefs.loc[val_attn_coefs.cv_iter == "cv_0"].sort_index()

In [ ]:
val_attn_coefs.loc[val_attn_coefs.cv_iter == "cv_1"].sort_index()

In [ ]:
## Site-to-site distance distribution
fig, ax = plt.subplots(figsize=(8, 6))

sns.histplot(val_attn_coefs["dist"], bins=20, ax=ax)

ax.set_xlabel("Site-to-site distance (km)")
ax.set_ylabel("Count")
ax.grid(linewidth=0.5, alpha=0.5, linestyle="--")

fig.tight_layout()
